# RSASE publication figure pipeline

This cleaned notebook contains the reproducible code used for Figures 2–10.
Figure 1 (the study-area map) is supplied as a finalized cartographic output.
Large raster inputs and separately licensed satellite panels are documented in the repository README files.

The 2013 water-index raster is treated as MNDWI; its original filename is retained.


## Figure 2 — LULC distribution


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import matplotlib.patheffects as pe
from matplotlib.gridspec import GridSpec
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from pyproj import Transformer
import contextily as ctx
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
FIGURE_DIR = REPO_ROOT / 'figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# ── 1. FILE PATHS ─────────────────────────────────────────────
FILES = {
    2002: str(REPO_ROOT / 'data' / 'raw' / 'lulc' / '2002LULC.tif'),
    2013: str(REPO_ROOT / 'data' / 'raw' / 'lulc' / '2013LULC.tif'),
    2022: str(REPO_ROOT / 'data' / 'raw' / 'lulc' / '2022LULC.tif'),
}
OUTPUT_PATH = str(FIGURE_DIR / 'Figure_02_LULC_Distribution.png')
DPI         = 600

# ── 2. CLASS DEFINITIONS ──────────────────────────────────────
CLASSES = {
    1: {'name': 'Waterbody',        'color': '#1529C7'},
    2: {'name': 'Bare Surface',     'color': '#C4C3C3'},
    3: {'name': 'Built-up',         'color': '#FF0403'},
    4: {'name': 'Dense Vegetation', 'color': '#224623'},
    5: {'name': 'Light Vegetation', 'color': '#8FE375'},
    6: {'name': 'Wetland',          'color': '#33A02C'},
}
CLASS_VALS = sorted(CLASSES.keys())
COLORS     = [CLASSES[v]['color'] for v in CLASS_VALS]
NAMES      = [CLASSES[v]['name']  for v in CLASS_VALS]
CMAP       = mcolors.ListedColormap(COLORS)
BOUNDS     = [v - 0.5 for v in CLASS_VALS] + [CLASS_VALS[-1] + 0.5]
NORM       = mcolors.BoundaryNorm(BOUNDS, CMAP.N)

# ── 3. LOAD & REPROJECT TO WEB MERCATOR (EPSG:3857) ──────────
# Contextily fetches tiles in EPSG:3857 — plotting in the same CRS
# ensures the basemap always renders correctly.
def load_to_mercator(path):
    dst_crs = 'EPSG:3857'
    with rasterio.open(path) as src:
        nodata = src.nodata
        transform, width, height = calculate_default_transform(
            src.crs, dst_crs, src.width, src.height, *src.bounds)
        data_dst = np.full((height, width), np.nan, dtype=np.float32)
        source_data = src.read(1).astype(np.float32)
        source_data[(source_data == nodata) | (source_data == -999)] = np.nan
        reproject(
            source=source_data,
            destination=data_dst,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=transform,
            dst_crs=dst_crs,
            resampling=Resampling.nearest,
            src_nodata=nodata,
            dst_nodata=np.nan,
        )
    left  = transform.c
    top   = transform.f
    right = left + transform.a * width
    bot   = top  + transform.e * height
    # extent = [left, right, bottom, top] in metres (Web Mercator)
    return data_dst, [left, right, bot, top]

rasters = {}
for year, path in FILES.items():
    arr, extent = load_to_mercator(path)
    rasters[year] = {'data': arr, 'extent': extent}

# ── 4. FIGURE SIZE ────────────────────────────────────────────
ext       = next(iter(rasters.values()))['extent']
map_w     = ext[1] - ext[0]   # metres
map_h     = ext[3] - ext[2]   # metres
aspect_hw = map_h / map_w     # already corrected in projected CRS

panel_w   = 8.5
panel_h   = panel_w * aspect_hw
fig_w     = panel_w * 2 + 0.8
fig_h     = panel_h * 2 + 1.8

# ── 5. TICK FORMATTERS (convert metres → lat/lon labels) ──────
_to_ll = Transformer.from_crs('EPSG:3857', 'EPSG:4326', always_xy=True)

def fmt_lon(x, _):
    lon, _ = _to_ll.transform(x, 0)
    return f"{abs(lon):.2f}°{'E' if lon >= 0 else 'W'}"

def fmt_lat(y, _):
    # use a representative x (centre of study area) for accuracy
    _, lat = _to_ll.transform(390000, y)
    return f"{abs(lat):.2f}°{'N' if lat >= 0 else 'S'}"

# ── 6. HELPERS ────────────────────────────────────────────────
def add_scalebar(ax, extent, length_frac=0.22, color='#222222'):
    x0, x1, y0, y1 = extent
    map_w_m  = x1 - x0
    map_h_m  = y1 - y0
    bar_m    = map_w_m * length_frac
    bar_km   = bar_m / 1000
    bar_km_r = max(1, round(bar_km / 5) * 5)
    bar_m_r  = bar_km_r * 1000
    xe = x1 - map_w_m * 0.04
    xs = xe - bar_m_r
    yp = y0 + map_h_m * 0.01
    th = map_h_m * 0.008
    shadow = [pe.withStroke(linewidth=3, foreground='white')]
    ax.plot([xs, xe], [yp, yp], color=color, lw=2.5,
            solid_capstyle='butt', zorder=12, path_effects=shadow)
    for xp in (xs, xe):
        ax.plot([xp, xp], [yp - th, yp + th], color=color,
                lw=1.5, zorder=12, path_effects=shadow)
    ax.text((xs + xe) / 2, yp + map_h_m * 0.022,
            f'{int(bar_km_r)} km',
            ha='center', va='bottom', fontsize=8,
            color=color, fontweight='bold', zorder=12,
            path_effects=[pe.withStroke(linewidth=2, foreground='white')])

def add_north_arrow(ax, extent, color='#222222'):
    x0, x1, y0, y1 = extent
    map_w_m   = x1 - x0
    map_h_m   = y1 - y0
    ax_x      = x0 + map_w_m * 0.917
    arrow_len = map_h_m * 0.07
    tip_y     = y0 + map_h_m * 0.88
    base_y    = tip_y - arrow_len
    shadow    = [pe.withStroke(linewidth=2.5, foreground='white')]
    ax.annotate('', xy=(ax_x, tip_y), xytext=(ax_x, base_y),
                arrowprops=dict(arrowstyle='-|>', color=color,
                                lw=2.2, mutation_scale=14), zorder=12)
    ax.text(ax_x, tip_y + map_h_m * 0.035, 'N',
            ha='center', va='bottom', fontsize=10,
            color=color, fontweight='bold', zorder=12,
            path_effects=shadow)

# ── 7. BUILD FIGURE ───────────────────────────────────────────
gs = GridSpec(
    2, 2,
    height_ratios=[panel_h, panel_h],
    hspace=0.10, wspace=0.08,
    left=0.07, right=0.99,
    top=0.94, bottom=0.04,
)
fig = plt.figure(figsize=(fig_w, fig_h), dpi=DPI, facecolor='white')
fig.patch.set_facecolor('white')
gs.figure = fig

ESRI_LIGHT_GREY = (
    'https://server.arcgisonline.com/ArcGIS/rest/services/'
    'Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}'
)

YEARS_GRID = [(2002, 0, 0), (2013, 0, 1), (2022, 1, 0)]

for year, row, col in YEARS_GRID:
    rd     = rasters[year]
    arr    = rd['data']
    extent = rd['extent']
    x0, x1, y0, y1 = extent

    ax = fig.add_subplot(gs[row, col])
    ax.set_facecolor('#F0F0F0')
    ax.set_xlim(x0, x1)
    ax.set_ylim(y0, y1)

    # ── Basemap in EPSG:3857 — always works with contextily ───
    try:
        ctx.add_basemap(
            ax,
            crs='EPSG:3857',
            source=ESRI_LIGHT_GREY,
            attribution=False,
            zoom='auto',
            zorder=1,
        )
        print(f"Basemap loaded for {year}")
    except Exception as e:
        print(f"Basemap failed ({year}): {e}")
        # Fallback to OpenStreetMap
        try:
            ctx.add_basemap(ax, crs='EPSG:3857',
                            source=ctx.providers.OpenStreetMap.Mapnik,
                            attribution=False, zoom='auto', zorder=1)
            print(f"OSM fallback loaded for {year}")
        except Exception as e2:
            print(f"All basemaps failed ({year}): {e2}")

    # ── LULC overlay — use RGBA so NaN pixels are transparent ─
    # Convert class array to RGBA manually so the basemap shows
    # through wherever there is no data (NaN) and at class edges
    rgba = np.zeros((*arr.shape, 4), dtype=np.float32)
    for val, cls in CLASSES.items():
        mask = (arr == val)
        rgb  = mcolors.to_rgb(cls['color'])
        rgba[mask, 0] = rgb[0]
        rgba[mask, 1] = rgb[1]
        rgba[mask, 2] = rgb[2]
        rgba[mask, 3] = 0.82    # opaque for valid class pixels
    # NaN pixels remain fully transparent (alpha=0) → basemap shows through

    ax.imshow(rgba,
              extent=[x0, x1, y0, y1], origin='upper',
              interpolation='nearest', aspect='equal',
              zorder=3)
    ax.set_xlim(x0, x1)
    ax.set_ylim(y0, y1)

    # Spines
    for spine in ax.spines.values():
        spine.set_edgecolor('#888888')
        spine.set_linewidth(0.8)

    # Ticks — display as lat/lon even though axes are in metres
    ax.xaxis.set_major_locator(mticker.MaxNLocator(4, prune='both'))
    ax.yaxis.set_major_locator(mticker.MaxNLocator(5, prune='both'))
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(fmt_lon))
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_lat))
    ax.tick_params(axis='both', labelsize=8, colors='#333333',
                   length=3.5, width=0.7)

    if row == 0:
        ax.tick_params(labelbottom=False)
    if col == 1:
        ax.tick_params(labelleft=False)
    if row == 1:
        ax.set_xlabel('Longitude', fontsize=9, color='#333333', labelpad=5)
    if col == 0:
        ax.set_ylabel('Latitude', fontsize=9, color='#333333', labelpad=5)

    # Year badge
    ax.text(0.03, 0.97, str(year),
            transform=ax.transAxes,
            fontsize=20, fontweight='bold', color='#111111',
            va='top', ha='left', zorder=14,
            bbox=dict(boxstyle='round,pad=0.35',
                      facecolor='white', edgecolor='#AAAAAA',
                      alpha=0.88, linewidth=0.9))

    add_north_arrow(ax, extent)
    add_scalebar(ax, extent)

# ── 8. BOTTOM-RIGHT: LULC CLASS LEGEND ────────────────────────
ax_legend = fig.add_subplot(gs[1, 1])
ax_legend.set_facecolor('white')
ax_legend.set_xlim(0, 1)
ax_legend.set_ylim(0, 1)
ax_legend.axis('off') # Hide axes for the legend panel

legend_patches = [
    mpatches.Patch(facecolor=CLASSES[v]['color'],
                   edgecolor='#555555', linewidth=0.5,
                   label=CLASSES[v]['name'])
    for v in CLASS_VALS
]

# Calculate optimal legend columns based on number of classes
num_classes = len(CLASSES)
if num_classes <= 3: # Up to 3 classes fit well in 1 column
    ncol = 1
elif num_classes <= 6: # 4-6 classes in 2 columns
    ncol = 2
else: # More than 6 classes in 3 columns
    ncol = 3

ax_legend.legend(
    handles=legend_patches,
    loc='center', # Center the legend in the panel
    ncol=ncol,
    fontsize=10, # Slightly larger font for readability
    frameon=True,
    framealpha=0.9,
    edgecolor='#CCCCCC',
    title='LULC Classes',
    title_fontsize=12, # Larger title for the legend
    bbox_to_anchor=(0.5, 0.5), # Anchor at center of the subplot
    bbox_transform=ax_legend.transAxes
)


# ── 9. MAIN TITLE ─────────────────────────────────────────────
fig.suptitle(
    'Land Use Land Cover (LULC) Distribution — 2002, 2013 & 2022',
    fontsize=14, fontweight='bold', color='#111111', y=0.975)

# ── 10. SAVE ──────────────────────────────────────────────────
fig.savefig(OUTPUT_PATH, dpi=DPI, bbox_inches='tight', facecolor='white')
plt.show()
print(f"Saved: {OUTPUT_PATH}")

# Uncomment to auto-download in Colab:
# from google.colab import files
# files.download(OUTPUT_PATH)

## Figure 3 — LST distribution


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as mticker
from matplotlib.colorbar import ColorbarBase
import matplotlib.patheffects as pe
from matplotlib.gridspec import GridSpec
from scipy.ndimage import gaussian_filter
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
import contextily as ctx
import warnings
warnings.filterwarnings('ignore')

# ── 1. CONFIGURATION ─────────────────────────────────────────
FILES = {
    2002: str(REPO_ROOT / 'data' / 'raw' / 'lst' / '2002_LST.tif'),
    2013: str(REPO_ROOT / 'data' / 'raw' / 'lst' / '2013_LST.tif'),
    2022: str(REPO_ROOT / 'data' / 'raw' / 'lst' / '2022_LST.tif'),
}
OUTPUT_PATH = str(FIGURE_DIR / 'Figure_03_LST_Distribution.png')
DPI = 600

# ── 2. COLOUR RAMP ───────────────────────────────────────────
LST_COLORS = [
    '#004500', '#1a6e1a', '#4ca64c', '#88cc88',
    '#c8e6c8', '#ffffbf', '#fee090', '#fdae61',
    '#f46d43', '#d73027', '#a50026',
]
LST_CMAP = mcolors.LinearSegmentedColormap.from_list(
    'LST_thermal', LST_COLORS, N=512)

# ── 3. LOAD & REPROJECT TO WGS84 ─────────────────────────────
def load_and_reproject(path, dst_crs='EPSG:4326'):
    with rasterio.open(path) as src:
        src_crs = src.crs
        nodata = src.nodata
        if src_crs.to_epsg() == 4326:
            data = src.read(1).astype(np.float32)
            bounds = src.bounds
            extent = [bounds.left, bounds.right, bounds.bottom, bounds.top]
            if nodata is not None:
                data = np.where(data == nodata, np.nan, data)
            data = np.where(data <= 0, np.nan, data)
            return data, extent
        transform, width, height = calculate_default_transform(
            src_crs, dst_crs, src.width, src.height, *src.bounds)
        data_dst = np.full((height, width), np.nan, dtype=np.float32)
        reproject(
            source=rasterio.band(src, 1),
            destination=data_dst,
            src_transform=src.transform,
            src_crs=src_crs,
            dst_transform=transform,
            dst_crs=dst_crs,
            resampling=Resampling.bilinear,
            src_nodata=nodata,
            dst_nodata=np.nan,
        )
    data_dst = np.where(data_dst <= 0, np.nan, data_dst)
    left = transform.c
    top = transform.f
    right = left + transform.a * width
    bot = top + transform.e * height
    return data_dst, [left, right, bot, top]


rasters = {}
for year, path in FILES.items():
    arr, extent = load_and_reproject(path)
    rasters[year] = {'data': arr, 'extent': extent}

all_p = [np.nanpercentile(r['data'], [2, 98]) for r in rasters.values()]
vmin = float(min(p[0] for p in all_p))
vmax = float(max(p[1] for p in all_p))

# ── 4. FIGURE SIZE FROM RASTER ASPECT RATIO ──────────────────
sample = next(iter(rasters.values()))
ext = sample['extent']
map_w = ext[1] - ext[0]
map_h = ext[3] - ext[2]
lat_mid = (ext[2] + ext[3]) / 2
cos_lat = np.cos(np.radians(lat_mid))
aspect_hw = map_h / (map_w * cos_lat)

panel_w = 8.5
panel_h = panel_w * aspect_hw
fig_w = panel_w * 2 + 0.8
fig_h = panel_h * 2 + 1.2

# ── 5. HELPERS ───────────────────────────────────────────────
def fmt_lon(v, _):
    return f"{abs(v):.2f}°{'E' if v >= 0 else 'W'}"


def fmt_lat(v, _):
    return f"{abs(v):.2f}°{'N' if v >= 0 else 'S'}"


def add_scalebar(ax, extent, length_frac=0.22, color='#222222'):
    x0, x1, y0, y1 = extent
    map_w_d = x1 - x0
    map_h_d = y1 - y0
    bar_deg = map_w_d * length_frac
    lat_mid = (y0 + y1) / 2
    bar_km = bar_deg * 111.32 * np.cos(np.radians(lat_mid))
    bar_km_r = max(1, round(bar_km / 5) * 5)
    xe = x1 - map_w_d * 0.04
    xs = xe - bar_deg
    yp = y0 + map_h_d * 0.01
    th = map_h_d * 0.012
    shadow = [pe.withStroke(linewidth=3, foreground='white')]
    ax.plot([xs, xe], [yp, yp], color=color, lw=2.5,
            solid_capstyle='butt', zorder=12, path_effects=shadow)
    for xp in (xs, xe):
        ax.plot([xp, xp], [yp - th, yp + th], color=color,
                lw=1.5, zorder=12, path_effects=shadow)
    ax.text((xs + xe) / 2, yp + map_h_d * 0.032,
            f'{int(bar_km_r)} km', ha='center', va='bottom', fontsize=8,
            color=color, fontweight='bold', zorder=12,
            path_effects=[pe.withStroke(linewidth=2, foreground='white')])


def add_north_arrow_and_stats(ax, arr, extent, color='#222222'):
    valid = arr[~np.isnan(arr)]
    t_min = np.percentile(valid, 2)
    t_mean = np.mean(valid)
    t_max = np.percentile(valid, 98)
    txt = (f'Min:  {t_min:.1f}°C\n'
           f'Mean: {t_mean:.1f}°C\n'
           f'Max:  {t_max:.1f}°C')
    ax.text(0.97, 0.97, txt, transform=ax.transAxes,
            fontsize=7.5, color='#111111', va='top', ha='right', zorder=13,
            linespacing=1.6, fontfamily='monospace',
            bbox=dict(boxstyle='round,pad=0.4', facecolor='white',
                      edgecolor='#AAAAAA', alpha=0.88, linewidth=0.7))

    x0, x1, y0, y1 = extent
    map_w_d = x1 - x0
    map_h_d = y1 - y0
    ax_x = x0 + map_w_d * 0.917
    arrow_len = map_h_d * 0.07
    stats_bot_ax = 0.74
    tip_ax = stats_bot_ax - 0.06
    base_ax = tip_ax - (arrow_len / map_h_d)
    tip_y = y0 + tip_ax * map_h_d
    base_y = y0 + base_ax * map_h_d

    shadow = [pe.withStroke(linewidth=2.5, foreground='white')]
    ax.annotate('', xy=(ax_x, tip_y), xytext=(ax_x, base_y),
                arrowprops=dict(arrowstyle='-|>', color=color,
                                lw=2.2, mutation_scale=14), zorder=12)
    ax.text(ax_x, tip_y + map_h_d * 0.035, 'N',
            ha='center', va='bottom', fontsize=10, color=color,
            fontweight='bold', zorder=12, path_effects=shadow)


# ── 6. BUILD 2×2 GRID ────────────────────────────────────────
YEARS_GRID = [
    (2002, 0, 0),
    (2013, 0, 1),
    (2022, 1, 0),
]

gs = GridSpec(
    2, 2, figure=None,
    height_ratios=[panel_h, panel_h],
    hspace=0.12, wspace=0.08,
    left=0.07, right=0.99, top=0.94, bottom=0.07,
)

fig = plt.figure(figsize=(fig_w, fig_h), dpi=DPI, facecolor='white')
gs.figure = fig

ESRI_LIGHT_GREY = (
    'https://server.arcgisonline.com/ArcGIS/rest/services/'
    'Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}'
)

map_axes = {}
for year, row, col in YEARS_GRID:
    rd = rasters[year]
    arr = rd['data']
    extent = rd['extent']
    x0, x1, y0, y1 = extent

    ax = fig.add_subplot(gs[row, col])
    map_axes[(row, col)] = ax
    ax.set_facecolor('white')
    ax.set_xlim(x0, x1)
    ax.set_ylim(y0, y1)

    try:
        ctx.add_basemap(ax, crs='EPSG:4326', source=ESRI_LIGHT_GREY,
                        attribution=False, zoom='auto', zorder=1)
    except Exception as e:
        print(f"Basemap failed ({year}): {e}")
        ax.set_facecolor('#EFEFEF')

    med = float(np.nanmedian(arr))
    display = gaussian_filter(np.where(np.isnan(arr), med, arr), sigma=0.6)
    display = np.where(np.isnan(arr), np.nan, display)

    ax.imshow(display, cmap=LST_CMAP, vmin=vmin, vmax=vmax,
              extent=[x0, x1, y0, y1], origin='upper',
              interpolation='bilinear', aspect='equal', alpha=0.82, zorder=3)
    ax.set_xlim(x0, x1)
    ax.set_ylim(y0, y1)

    for spine in ax.spines.values():
        spine.set_edgecolor('#888888')
        spine.set_linewidth(0.8)

    ax.xaxis.set_major_locator(mticker.MaxNLocator(4, prune='both'))
    ax.yaxis.set_major_locator(mticker.MaxNLocator(5, prune='both'))
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(fmt_lon))
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_lat))
    ax.tick_params(axis='both', labelsize=8, colors='#333333',
                   length=3.5, width=0.7)

    if row == 0:
        ax.tick_params(labelbottom=False)
    if col == 1:
        ax.tick_params(labelleft=False)
    if row == 1:
        ax.set_xlabel('Longitude', fontsize=9, color='#333333', labelpad=5)
    if col == 0:
        ax.set_ylabel('Latitude', fontsize=9, color='#333333', labelpad=5)

    ax.text(0.03, 0.97, str(year), transform=ax.transAxes,
            fontsize=20, fontweight='bold', color='#111111',
            va='top', ha='left', zorder=14,
            bbox=dict(boxstyle='round,pad=0.35', facecolor='white',
                      edgecolor='#AAAAAA', alpha=0.88, linewidth=0.9))

    add_north_arrow_and_stats(ax, arr, extent)
    add_scalebar(ax, extent)

# ── 7. COMPACT BOTTOM-RIGHT LEGEND ───────────────────────────
# Keep the empty grid cell as a container, and place a small colorbar inside it.
# inset_axes coordinates are relative to this panel: [left, bottom, width, height].
ax_legend_panel = fig.add_subplot(gs[1, 1])
ax_legend_panel.set_axis_off()

# A subtle card keeps the legend visually grouped without filling the panel.
legend_card = plt.Rectangle(
    (0.25, 0.13), 0.50, 0.74,
    transform=ax_legend_panel.transAxes,
    facecolor='#FAFAFA', edgecolor='#D0D0D0', linewidth=0.8,
    zorder=0,
)
ax_legend_panel.add_patch(legend_card)

ax_legend_panel.text(
    0.50, 0.81, 'LST Color Scale',
    transform=ax_legend_panel.transAxes,
    fontsize=11, fontweight='bold', color='#111111',
    va='center', ha='center',
)

# Narrow, centered vertical colorbar; it no longer inherits the whole subplot.
cax = ax_legend_panel.inset_axes([0.43, 0.24, 0.075, 0.48])
cbar = ColorbarBase(
    cax,
    cmap=LST_CMAP,
    norm=mcolors.Normalize(vmin=vmin, vmax=vmax),
    orientation='vertical',
    ticklocation='right',
)
cbar.locator = mticker.MaxNLocator(nbins=6)
cbar.update_ticks()
cbar.set_label('Land Surface Temperature (°C)',
               fontsize=9, color='#222222', labelpad=7)
cbar.ax.tick_params(labelsize=8, colors='#222222', length=3, width=0.8)
cbar.outline.set_edgecolor('#888888')
cbar.outline.set_linewidth(0.8)

# ── 8. TITLE ─────────────────────────────────────────────────
fig.suptitle(
    'Land Surface Temperature (LST) Distribution — 2002, 2013 & 2022',
    fontsize=14, fontweight='bold', color='#111111', y=0.975)

plt.subplots_adjust(left=0.07, right=0.99, top=0.95,
                    bottom=0.07, wspace=0.07)

# ── 9. SAVE ──────────────────────────────────────────────────
fig.savefig(OUTPUT_PATH, dpi=DPI, bbox_inches='tight', facecolor='white')
plt.show()
print(f"Saved: {OUTPUT_PATH}")

# Uncomment to auto-download in Colab:
# from google.colab import files
# files.download(OUTPUT_PATH)


## Figure 4 — LST correlations with NDVI, NDBI and MNDWI


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

# ── 1. LOAD DATA ───────────────────
# Option A: Upload the file manually in Colab
#   from google.colab import files
#   uploaded = files.upload()  # then set FILE_PATH to the filename

# Option B: Mount Google Drive
#   from google.drive import drive
#   drive.mount('/content/drive')
#   FILE_PATH = '/content/drive/MyDrive/New_Updated_Zonal_Statistics.xlsx'

FILE_PATH = str(REPO_ROOT / 'data' / 'derived' / 'New Updated Zonal Statistics.xlsx')
OUTPUT_PATH = str(FIGURE_DIR / 'Figure_04_LST_Spectral_Correlations.png')

df = pd.read_excel(FILE_PATH)

# ── 2. COLUMN MAPPING ────────────────────
years     = [2002,        2013,        2022       ]
lst_cols  = ['_2002LST_m','_2013LST_m','_2022LST_m']
ndvi_cols = ['_NDVI2002_','_NDVI2013_','_NDVI2022_']
ndbi_cols = ['_NDBI2002_','_NDBI2013_','_NDBI2022_']
ndwi_cols = ['_MNDWI2002','_MNDWI2013','_MNDWI2022']

index_labels    = ['NDVI',   'NDBI',   'MNDWI'  ]
index_cols_list = [ndvi_cols, ndbi_cols, ndwi_cols]

# ── 3. STYLE SETTINGS ────────────────────
year_colors = {2002: '#2166AC', 2013: '#1A9641', 2022: '#D7191C'}
reg_colors  = {2002: '#053061', 2013: '#005824', 2022: '#67000D'}

# ── 4. BUILD FIGURE ─────────────────────
fig = plt.figure(figsize=(16, 14))
fig.patch.set_facecolor('#F8F9FA')

gs = fig.add_gridspec(
    3, 3,
    hspace=0.42, wspace=0.35,
    left=0.07, right=0.97, top=0.94, bottom=0.06
)

for row, (year, lst_col) in enumerate(zip(years, lst_cols)):
    for col, (idx_label, idx_cols) in enumerate(zip(index_labels, index_cols_list)):
        ax = fig.add_subplot(gs[row, col])
        ax.set_facecolor('white')

        # ── Clean & align data ──
        x_series = df[idx_cols[row]].dropna()
        y_series = df[lst_col][x_series.index].dropna()
        common   = x_series.index.intersection(y_series.index)
        x, y     = x_series[common].values, y_series[common].values

        # ── Linear regression ──
        slope, intercept, r, p, se = stats.linregress(x, y)
        r2     = r ** 2
        x_line = np.linspace(x.min(), x.max(), 200)
        y_line = slope * x_line + intercept

        # ── Plot scatter and regression line ──
        ax.scatter(x, y, s=10, alpha=0.6, color=year_colors[year], zorder=2)
        ax.plot(x_line, y_line, color=reg_colors[year], linewidth=1.5, zorder=3)

        # ── Equation annotation ──
        sign    = '+' if intercept >= 0 else '-'
        eq_text = f'y = {slope:.3f}x {sign} {abs(intercept):.3f}\nr = {r:.4f}' # Changed R² to r
        ax.text(0.97, 0.97, eq_text,
                transform=ax.transAxes,
                fontsize=8.5, va='top', ha='right',
                bbox=dict(boxstyle='round,pad=0.4',
                          facecolor='white', edgecolor='#CCCCCC', alpha=0.9))

        # ── Labels & formatting ──
        ax.set_title(f'{year} LST vs {idx_label}',
                     fontsize=11, fontweight='bold', color='#222222', pad=8)
        ax.set_xlabel(f'Mean {idx_label}',
                      fontsize=9.5, color='#444444', labelpad=4)
        ax.set_ylabel('Mean LST (°C)',
                      fontsize=9.5, color='#444444', labelpad=4)
        ax.grid(True, linestyle='--', linewidth=0.5,
                color='#DDDDDD', alpha=0.8, zorder=1)
        ax.set_axisbelow(True)
        for spine in ax.spines.values():
            spine.set_edgecolor('#BBBBBB')
            spine.set_linewidth(0.8)
        ax.tick_params(axis='both', labelsize=8.5, colors='#555555', length=3)

# ── Year labels on left margin ──
for row, year in enumerate(years):
    ax0 = fig.axes[row * 3]
    pos = ax0.get_position()
    fig.text(0.005, pos.y0 + pos.height / 2,
             str(year), va='center', ha='left',
             fontsize=13, fontweight='bold',
             color=year_colors[year], rotation=90)

fig.suptitle('Correlation of Mean LST with Spectral Indices (2002–2022)',
             fontsize=14, fontweight='bold', color='#111111', y=0.975)

# ── 5. SAVE & SHOW ─────────────────────
plt.savefig(OUTPUT_PATH, dpi=600, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print(f"Figure saved to: {OUTPUT_PATH}")

# ── Optional: auto-download in Colab ─────────
# from google.colab import files
# files.download(OUTPUT_PATH)

## Figures 5–8 — Fixed corridor composites re-exported at 500 dpi


In [ ]:
# ============================================================
# LST Corridor Composites — All 4 Transects at 500 DPI
# Oshodi–Ikeja GRA | Ajeromi–Apapa | Makoko–UNILAG | Obalende–Ikoyi
# Compatible with Google Colab
# ============================================================
# STEP 1 — Install dependencies (run once):
#   !pip install pandas matplotlib numpy scipy openpyxl pillow
#
# STEP 2 — Upload all required files listed in CORRIDORS below
#
# STEP 3 — Run this script — saves 4 PNG files at 500 DPI
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
from scipy.ndimage import gaussian_filter1d
from PIL import Image
import math, warnings
warnings.filterwarnings('ignore')

Image.MAX_IMAGE_PIXELS = None   # allow large satellite images

# ── 1. GLOBAL SETTINGS ───────────────────────────────────────
EXCEL     = str(REPO_ROOT / 'data' / 'derived' / 'Profile Graph_2002_2023.xlsx')
DPI       = 500
SMOOTH    = 2.5
M_PER_DEG = math.cos(math.radians(6.0)) * 111_320
YEARS     = [2002, 2013, 2022]

YEAR_STYLES = {
    2002: {'color': '#D85A30', 'lw': 2.2, 'ls': '-',  'alpha': 0.95, 'zorder': 4},
    2013: {'color': '#639922', 'lw': 2.0, 'ls': '--', 'alpha': 0.90, 'zorder': 3},
    2022: {'color': '#185FA5', 'lw': 2.0, 'ls': ':',  'alpha': 0.95, 'zorder': 5},
}

# ── 2. CORRIDOR DEFINITIONS ──────────────────────────────────
CORRIDORS = [
    {
        'title':         'Oshodi-Ikeja GRA: LST Spatial Distribution & Transect Profile (2002, 2013 & 2022)',
        'profile_title': 'LST Profile - Oshodi to Ikeja GRA (A to B)',
        'sheet':         'Sheet1',
        'map_2002':      str(REPO_ROOT / 'assets' / 'profile_maps' / 'Oshodi to Ikeja 2002 LST Profile Map.png'),
        'map_2013':      str(REPO_ROOT / 'assets' / 'profile_maps' / 'Oshodi to Ikeja 2013 LST Profile Map.png'),
        'map_2022':      str(REPO_ROOT / 'assets' / 'profile_maps' / 'Oshodi to Ikeja 2022 LST Profile Map.png'),
        'satellite':     str(REPO_ROOT / 'assets' / 'satellite_images' / 'Oshodi-Ikeja GRA2.png'),
        'output':        str(FIGURE_DIR / 'Figure_06_Oshodi_Ikeja_Corridor.png'),
    },
    {
        'title':         'Ajeromi-Apapa: LST Spatial Distribution & Transect Profile (2002, 2013 & 2022)',
        'profile_title': 'LST Profile - Ajegunle to Apapa (A to B)',
        'sheet':         'Sheet2',
        'map_2002':      str(REPO_ROOT / 'assets' / 'profile_maps' / 'Ajegunle to Apapa 2002 LST Profile Map.png'),
        'map_2013':      str(REPO_ROOT / 'assets' / 'profile_maps' / 'Ajegunle to Apapa 2013 LST Profile Map.png'),
        'map_2022':      str(REPO_ROOT / 'assets' / 'profile_maps' / 'Ajegunle to Apapa 2022 LST Profile Map.png'),
        'satellite':     str(REPO_ROOT / 'assets' / 'satellite_images' / 'Ajeromi-Apapa Satellite Imagery.png'),
        'output':        str(FIGURE_DIR / 'Figure_05_Ajeromi_Apapa_Corridor.png'),
    },
    {
        'title':         'Makoko-UNILAG: LST Spatial Distribution & Transect Profile (2002, 2013 & 2022)',
        'profile_title': 'LST Profile - Makoko to UNILAG/Abule-Oja (A to B)',
        'sheet':         'Sheet3',
        'map_2002':      str(REPO_ROOT / 'assets' / 'profile_maps' / 'Makoko to UNILAG 2002 LST Profile Map.png'),
        'map_2013':      str(REPO_ROOT / 'assets' / 'profile_maps' / 'Makoko to UNILAG 2013 LST Profile Map.png'),
        'map_2022':      str(REPO_ROOT / 'assets' / 'profile_maps' / 'Makoko to UNILAG 2022 LST Profile Map.png'),
        'satellite':     str(REPO_ROOT / 'assets' / 'satellite_images' / 'Abule Oja - Makoko2.png'),
        'output':        str(FIGURE_DIR / 'Figure_07_Makoko_UNILAG_Corridor.png'),
    },
    {
        'title':         'Obalende-Ikoyi: LST Spatial Distribution & Transect Profile (2002, 2013 & 2022)',
        'profile_title': 'LST Profile - Obalende to Ikoyi/Falomo (A to B)',
        'sheet':         'Sheet4',
        'map_2002':      str(REPO_ROOT / 'assets' / 'profile_maps' / 'Obalende to Ikoyi 2002 LST Profile Map.png'),
        'map_2013':      str(REPO_ROOT / 'assets' / 'profile_maps' / 'Obalende to Ikoyi 2013 LST Profile Map.png'),
        'map_2022':      str(REPO_ROOT / 'assets' / 'profile_maps' / 'Obalende to Ikoyi 2022 LST Profile Map.png'),
        'satellite':     str(REPO_ROOT / 'assets' / 'satellite_images' / 'Obalende-Ikoyi Satellite Imagery2.png'),
        'output':        str(FIGURE_DIR / 'Figure_08_Obalende_Ikoyi_Corridor.png'),
    },
]

# ── 3. DATA PARSER ───────────────────────────────────────────
def parse_sheet(df):
    header_row = None
    for i, row in df.iterrows():
        vals = set(row.dropna().values)
        if 2002.0 in vals and 2013.0 in vals and 2022.0 in vals:
            header_row = i
            break
    if header_row is None:
        header_row = 0
    col_map = {}
    for col in df.columns:
        val = df.loc[header_row, col]
        if val in (2002.0, 2013.0, 2022.0):
            col_map[int(val)] = col
    dist_col = None
    for col in df.columns:
        s = pd.to_numeric(df[col], errors='coerce').dropna()
        if len(s) > 10 and s.between(0, 0.2).mean() > 0.7:
            dist_col = col
            break
    data_rows = df.loc[header_row + 1:].copy()
    dist_deg  = pd.to_numeric(data_rows[dist_col], errors='coerce')
    mask      = dist_deg.notna() & dist_deg.between(0, 0.2)
    dist_m    = dist_deg[mask].values * M_PER_DEG
    result    = {'dist_m': dist_m}
    for year, col in col_map.items():
        result[year] = pd.to_numeric(
            data_rows.loc[mask.index[mask], col], errors='coerce').values
    return result

# Load all sheets once
xl_data = pd.read_excel(EXCEL, sheet_name=None, header=None)

# ── 4. COMPOSITE BUILDER ─────────────────────────────────────
def build_composite(corridor):
    print(f"Building: {corridor['output']} ...")

    dat = parse_sheet(xl_data[corridor['sheet']])
    x   = dat['dist_m']

    def load_for_panel(path):
        # Match the raster to its final 600 dpi panel dimensions while
        # preserving the source image content and aspect ratio.
        with Image.open(path) as source_image:
            panel_image = source_image.convert('RGB')
            panel_image.thumbnail((3000, 2400), Image.Resampling.LANCZOS)
            return np.array(panel_image)

    img_2002 = load_for_panel(corridor['map_2002'])
    img_2013 = load_for_panel(corridor['map_2013'])
    img_2022 = load_for_panel(corridor['map_2022'])
    img_sat  = load_for_panel(corridor['satellite'])

    # Panel height calculations
    map_ar = img_2002.shape[1] / img_2002.shape[0]
    sat_ar = img_sat.shape[1]  / img_sat.shape[0]
    fig_w  = 12.0
    col_w  = fig_w / 2
    map_h  = col_w / map_ar
    # Satellite panel height = its true height at col_w width (no stretching)
    sat_h  = col_w / sat_ar
    prof_h = max(2 * map_h - sat_h, map_h * 0.9)
    # Total figure height: title + 3 maps on left; right col = prof + sat + blank space
    fig_h  = 3 * map_h + 0.8

    fig = plt.figure(figsize=(fig_w, fig_h), dpi=DPI, facecolor='white')
    fig.patch.set_facecolor('white')

    outer = gridspec.GridSpec(
        1, 2, figure=fig,
        width_ratios=[1, 1], wspace=0.03,
        left=0.01, right=0.99, top=0.93, bottom=0.07,
    )
    left_gs = gridspec.GridSpecFromSubplotSpec(
        3, 1, subplot_spec=outer[0], hspace=0.01,
    )
    right_gs = gridspec.GridSpecFromSubplotSpec(
        2, 1, subplot_spec=outer[1],
        height_ratios=[prof_h, sat_h], hspace=0.01,
    )

    # Left column: LST maps
    for img_arr, row in [(img_2002, 0), (img_2013, 1), (img_2022, 2)]:
        ax = fig.add_subplot(left_gs[row, 0])
        ax.imshow(img_arr, aspect='auto')
        ax.axis('off')
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_edgecolor('#888888')
            spine.set_linewidth(0.6)

    # Right top: Profile line graph
    ax_p = fig.add_subplot(right_gs[0])
    ax_p.set_facecolor('#F7F9FC')

    smoothed = {}
    for year in YEARS:
        s = gaussian_filter1d(dat[year], sigma=SMOOTH)
        smoothed[year] = s
        st = YEAR_STYLES[year]
        ax_p.plot(x, s,
                  color=st['color'], linewidth=st['lw'],
                  linestyle=st['ls'], alpha=st['alpha'],
                  zorder=st['zorder'])

    s02, s22 = smoothed[2002], smoothed[2022]
    ax_p.fill_between(x, s02, s22, where=(s22 >= s02),
                      color='#D85A30', alpha=0.09, zorder=1)
    ax_p.fill_between(x, s02, s22, where=(s22 < s02),
                      color='#185FA5', alpha=0.10, zorder=1)

    all_vals = np.concatenate([dat[y] for y in YEARS])
    ax_ymin  = np.nanpercentile(all_vals, 0.5) - 0.5
    ax_ymax  = np.nanpercentile(all_vals, 99.5) + 1.0

    for year in YEARS:
        mv = float(np.mean(dat[year]))
        st = YEAR_STYLES[year]
        ax_p.axhline(mv, color=st['color'], lw=0.85,
                     linestyle='-.', alpha=0.50, zorder=2)
        ax_p.text(x[-1] * 1.006, mv, f'{mv:.1f}C',
                  color=st['color'], fontsize=7.5,
                  va='center', ha='left', fontweight='500', clip_on=False)

    for xval, lbl in [(x[0], 'A'), (x[-1], 'B')]:
        ax_p.axvline(xval, color='#333333', lw=1.0, ls='--', alpha=0.5, zorder=2)
        ax_p.text(xval, ax_ymax, lbl,
                  ha='center', va='top',
                  fontsize=11, fontweight='bold', color='white', zorder=10,
                  bbox=dict(boxstyle='circle,pad=0.4', facecolor='#333333',
                            edgecolor='white', linewidth=1.2, alpha=0.92))

    ax_p.annotate('',
        xy=(x[-1] * 0.88, ax_ymin + 0.08),
        xytext=(x[0] + x[-1] * 0.10, ax_ymin + 0.08),
        arrowprops=dict(arrowstyle='->', color='#666666',
                        lw=1.0, mutation_scale=10),
        zorder=8, clip_on=False)
    ax_p.text(x[-1] * 0.50, ax_ymin + 0.22,
              'Transect direction  A to B',
              ha='center', va='bottom',
              fontsize=7.5, color='#666666', style='italic')

    delta = float(np.mean(dat[2022]) - np.mean(dat[2002]))
    sign  = '+' if delta >= 0 else ''
    clr   = '#D85A30' if delta >= 0 else '#185FA5'
    ax_p.text(0.98, 0.97, f'Delta 2002 to 2022: {sign}{delta:.2f}C',
              transform=ax_p.transAxes,
              fontsize=9, color=clr, fontweight='bold',
              va='top', ha='right', zorder=10,
              bbox=dict(boxstyle='round,pad=0.35', facecolor='white',
                        edgecolor='#CCCCCC', alpha=0.92, linewidth=0.7))

    legend_els = [
        Line2D([0],[0], color=YEAR_STYLES[y]['color'], lw=2.2,
               linestyle=YEAR_STYLES[y]['ls'], alpha=YEAR_STYLES[y]['alpha'],
               label=str(y))
        for y in YEARS
    ] + [
        mpatches.Patch(facecolor='#D85A30', alpha=0.25, edgecolor='none',
                       label='Warming (2022 > 2002)'),
        mpatches.Patch(facecolor='#185FA5', alpha=0.25, edgecolor='none',
                       label='Cooling (2022 < 2002)'),
        Line2D([0],[0], color='#888888', lw=0.9, linestyle='-.',
               alpha=0.6, label='Per-year mean LST'),
    ]
    ax_p.legend(handles=legend_els, loc='upper right',
                fontsize=8, frameon=True, framealpha=0.95,
                edgecolor='#CCCCCC', ncol=1,
                bbox_to_anchor=(0.98, 0.88))

    ax_p.grid(True, linestyle='--', linewidth=0.45,
              color='#DDDDEE', alpha=1.0, zorder=0)
    ax_p.set_axisbelow(True)
    for sp in ['top', 'right']:
        ax_p.spines[sp].set_visible(False)
    for sp in ['left', 'bottom']:
        ax_p.spines[sp].set_color('#AAAAAA')
        ax_p.spines[sp].set_linewidth(0.7)

    ax_p.set_xlim(x[0], x[-1])
    ax_p.set_ylim(ax_ymin, ax_ymax)
    ax_p.xaxis.set_major_locator(mticker.MaxNLocator(6, prune='both'))
    ax_p.xaxis.set_major_formatter(
        mticker.FuncFormatter(lambda v, _: f'{v:.0f} m'))
    ax_p.yaxis.set_major_locator(mticker.MultipleLocator(1))
    ax_p.yaxis.set_minor_locator(mticker.MultipleLocator(0.5))
    ax_p.tick_params(axis='both', labelsize=9, colors='#444444',
                     length=3, width=0.6)
    ax_p.set_xlabel('Distance along transect (m)', fontsize=10,
                    color='#333333', labelpad=5)
    ax_p.set_ylabel('Mean LST (degrees C)', fontsize=10,
                    color='#333333', labelpad=5)
    # Title placed via fig.text so it doesn't add internal padding that
    # would push the plot area down below the top of the LST maps
    ax_p.set_title('')   # no internal title

    # Right bottom: satellite image — true aspect ratio, no stretching
    ax_s = fig.add_subplot(right_gs[1])
    h_sat, w_sat = img_sat.shape[:2]
    ax_s.imshow(img_sat, aspect='equal', extent=[0, w_sat, 0, h_sat])
    ax_s.set_xlim(0, w_sat)
    ax_s.set_ylim(0, h_sat)
    ax_s.axis('off')
    for spine in ax_s.spines.values():
        spine.set_visible(True)
        spine.set_edgecolor('#888888')
        spine.set_linewidth(0.6)
    # Label in data coordinates so it stays inside the image bounds
    ax_s.text(w_sat * 0.012, h_sat * 0.97, 'Satellite Imagery',
              fontsize=9, fontweight='bold', color='#111111',
              va='top', ha='left', zorder=5,
              bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                        edgecolor='#888888', alpha=0.80, linewidth=0.6))

    fig.suptitle(corridor['title'],
                 fontsize=14, fontweight='bold',
                 color='#111111', y=0.975)

    # Profile title placed INSIDE the axes, shifted right to clear the A marker
    ax_p.text(0.08, 0.99, corridor['profile_title'],
              transform=ax_p.transAxes,
              fontsize=10, fontweight='bold', color='#1A1A2E',
              va='top', ha='left', zorder=11)

    fig.savefig(corridor['output'], dpi=DPI,
                bbox_inches='tight', facecolor='white')
    plt.close(fig)
    print(f"  Saved: {corridor['output']}")


# ── 5. RUN ALL FOUR CORRIDORS ────────────────────────────────
for corridor in CORRIDORS:
    build_composite(corridor)

print('\nAll 4 corridor composites saved at 500 DPI.')

# Uncomment to auto-download all outputs in Colab:
# from google.colab import files
# for c in CORRIDORS:
#     files.download(c['output'])

## Figure 9 — Ward-level LST change


In [ ]:
# ============================================================
# LST Ward-Level Grouped Bar Charts — Sheet2 (4 Study Areas)
# Compatible with Google Colab
# ============================================================
# STEP 1 — Install dependencies (run once):
#   !pip install pandas matplotlib openpyxl
#
# STEP 2 — Upload LST_Statistics_summary.xlsx to Colab or
#           adjust FILE_PATH below
#
# STEP 3 — Run this script
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# ── 1. CONFIGURATION ─────────────────────────────────────────
FILE_PATH   = str(REPO_ROOT / 'data' / 'derived' / 'LST Statistics summary.xlsx')
OUTPUT_PATH = str(FIGURE_DIR / 'Figure_09_Ward_LST_Change.png')
DPI         = 600

# ── 2. DATA — read directly from Sheet2 ──────────────────────
table = pd.read_excel(FILE_PATH, sheet_name='Sheet2', header=3, usecols='E:I')
table.columns = ['Area', 'Ward', 2002, 2013, 2022]
table = table.dropna(subset=['Ward']).copy()
table['Area'] = table['Area'].ffill()

def display_ward(name):
    replacements = {
        'Marine Breach': 'Marine Beach',
        'Aderupoko/Ijebu Quarters': 'Aderupoko/\nIjebu Qtrs',
        'Akoka/Anu-Oluwapo': 'Akoka/\nAnu-Oluwapo',
        'Falomo/Oyinkan Abayomi': 'Falomo/\nOyinkan Abayomi',
        'Ijeh/Dolphin Estate': 'Ijeh/\nDolphin Estate',
        'Makoko Waterside': 'Makoko\nWaterside',
        'Onike/Oyadiran': 'Onike/\nOyadiran',
        'Owode/Orile Bariga': 'Owode/\nOrile Bariga',
        'Salami/Baiyewunmi': 'Salami/\nBaiyewunmi',
    }
    return replacements.get(str(name), str(name))

AREAS = []
for area_name, group in table.groupby('Area', sort=False):
    AREAS.append({
        'title': area_name,
        'wards': [display_ward(v) for v in group['Ward']],
        2002: group[2002].astype(float).tolist(),
        2013: group[2013].astype(float).tolist(),
        2022: group[2022].astype(float).tolist(),
    })

# ── 3. STYLE SETTINGS ────────────────────────────────────────
YEARS        = [2002, 2013, 2022]
COLORS       = ['#185FA5', '#639922', '#D85A30']   # blue, green, red-orange
EDGE_COLORS  = ['#042C53', '#27500A', '#4A1B0C']
HATCHES      = ['', '///', '...']                  # secondary visual cue per year

BAR_WIDTH    = 0.25
GROUP_PAD    = 0.10                                # extra space between ward groups
AREA_COLORS  = ['#E6F1FB', '#EAF3DE', '#FAEEDA', '#FAECE7']  # subtle panel tints

# ── 4. BUILD FIGURE ──────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(18, 12), dpi=DPI, facecolor='white')
fig.patch.set_facecolor('white')
axes = axes.flatten()

for ax_idx, (ax, area) in enumerate(zip(axes, AREAS)):
    wards  = area['wards']
    n      = len(wards)
    x      = np.arange(n)

    # ── Draw bars ────────────────────────────────────────────
    offsets = [-BAR_WIDTH, 0, BAR_WIDTH]
    for i, (year, color, edge, hatch, offset) in enumerate(
            zip(YEARS, COLORS, EDGE_COLORS, HATCHES, offsets)):
        ax.bar(
            x + offset,
            area[year],
            width=BAR_WIDTH,
            color=color,
            edgecolor=edge,
            linewidth=0.6,
            hatch=hatch,
            alpha=0.88,
            label=str(year),
            zorder=3,
        )

    # ── Panel background tint ─────────────────────────────────
    ax.set_facecolor(AREA_COLORS[ax_idx])

    # ── Grid ─────────────────────────────────────────────────
    ax.yaxis.grid(True, linestyle='--', linewidth=0.5,
                  color='white', alpha=0.9, zorder=1)
    ax.set_axisbelow(True)

    # ── Axis config ───────────────────────────────────────────
    ax.set_xticks(x)
    ax.set_xticklabels(wards, fontsize=7.5, ha='center', rotation=0)
    ax.set_ylim(25, 36.5)
    ax.yaxis.set_major_locator(plt.MultipleLocator(2))
    ax.yaxis.set_minor_locator(plt.MultipleLocator(1))
    ax.tick_params(axis='y', labelsize=8, colors='#333333')
    ax.tick_params(axis='x', length=0)
    ax.set_ylabel('Mean LST (°C)', fontsize=9, color='#333333', labelpad=6)

    # ── Spines ───────────────────────────────────────────────
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)
    for spine in ['left', 'bottom']:
        ax.spines[spine].set_color('#AAAAAA')
        ax.spines[spine].set_linewidth(0.7)

    # ── Panel title ───────────────────────────────────────────
    ax.set_title(area['title'],
                 fontsize=12, fontweight='bold',
                 color='#1A1A2E', pad=10, loc='left')

    # ── Value labels on bars ──────────────────────────────────
    for i, (year, offset) in enumerate(zip(YEARS, offsets)):
        for j, val in enumerate(area[year]):
            ax.text(
                j + offset, val + 0.08,
                f'{val:.1f}',
                ha='center', va='bottom',
                fontsize=5.5, color='#333333',
                fontweight='500',
            )

# ── 5. SHARED LEGEND ─────────────────────────────────────────
legend_patches = [
    mpatches.Patch(facecolor=COLORS[i], edgecolor=EDGE_COLORS[i],
                   hatch=HATCHES[i], linewidth=0.6, alpha=0.88,
                   label=str(yr))
    for i, yr in enumerate(YEARS)
]
fig.legend(
    handles=legend_patches,
    loc='lower center',
    ncol=3,
    frameon=True,
    framealpha=0.95,
    edgecolor='#CCCCCC',
    fontsize=10,
    title='Year',
    title_fontsize=10,
    bbox_to_anchor=(0.5, 0.01),
)

# ── 6. MAIN TITLE ─────────────────────────────────────────────
fig.suptitle(
    'Mean Land Surface Temperature (LST) by Ward — Selected Study Areas, Lagos (2002–2022)',
    fontsize=13, fontweight='bold', color='#111111', y=0.98,
)

plt.tight_layout(rect=[0, 0.06, 1, 0.96])

# ── 7. SAVE ───────────────────────────────────────────────────
fig.savefig(OUTPUT_PATH, dpi=DPI, bbox_inches='tight', facecolor='white')
plt.show()
print(f"Saved: {OUTPUT_PATH}")

# Uncomment to auto-download in Colab:
# from google.colab import files
# files.download(OUTPUT_PATH)

## Figure 10 — Socio-environmental radar profiles


In [ ]:
# ============================================================
# Ward Socio-Environmental Profiles — Radar (Spider) Chart
# Compatible with Google Colab
# ============================================================
# STEP 1 — Install dependencies (run once):
#   !pip install matplotlib numpy openpyxl pandas
#
# STEP 2 — Upload your Excel file in Colab:
#   from google.colab import files
#   uploaded = files.upload()   # select Ward_Socio_Economic_Data.xlsx
#
# STEP 3 — Run this script
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# ── 1. LOAD DATA ─────────────────────────────────────────────
FILE_PATH   = str(REPO_ROOT / 'data' / 'derived' / 'Ward_Socio_Economic_Data.xlsx')
OUTPUT_PATH = str(FIGURE_DIR / 'Figure_10_Ward_Socio_Environmental_Profiles.png')

df = pd.read_excel(FILE_PATH)

wards = []
for _, row in df.iterrows():
    wards.append({
        'name':   row['Ward'],
        'status': row['Planning Status'],
        'lst':    row['Mean LST (°C)'],
        'ndvi':   row['NDVI'],
        'ndbi':   row['NDBI'],
        'mndwi':  row['MNDWI'],
        'bd':     row['Building Density'],
        'pd':     row['Population Density (person/km²)'],
    })

# ── 2. AXIS CONFIGURATION ────────────────────────────────────
axes_keys   = ['lst',      'ndvi', 'ndbi',  'mndwi',
               'bd',               'pd']
axes_labels = ['LST (°C)', 'NDVI', 'NDBI',  'MNDWI\n(inverted)',
               'Building\nDensity', 'Population\nDensity']

mins = {'lst': 27,  'ndvi': 0,    'ndbi': -0.10, 'mndwi': -0.17, 'bd': 0,   'pd': 0}
maxs = {'lst': 34,  'ndvi': 0.22, 'ndbi':  0.10, 'mndwi':  0.00, 'bd': 0.6, 'pd': 60000}

def fmt(val, key):
    """Format raw value for vertex label."""
    if key == 'pd':
        return f'{int(round(val / 1000))}k'
    elif key == 'lst':
        return f'{val:.1f}'
    elif key == 'bd':
        return f'{val:.2f}'
    else:
        return f'{val:.3f}'

def norm(val, key):
    """Normalise to [0.05, 1]; invert MNDWI so further out = worse."""
    n = (val - mins[key]) / (maxs[key] - mins[key])
    if key == 'mndwi':
        n = 1 - n
    return max(0.05, min(1.0, n))

# ── 3. COLOUR PALETTE ────────────────────────────────────────
STATUS_COLOR = {
    'Planned':   '#185FA5',
    'Unplanned': '#D85A30',
    'Mixed':     '#3B6D11',
}
WARD_COLORS = {
    'Ikeja GRA':           '#1874CD',
    'Oduduwa (Apapa)':     '#4EA8E0',
    'Abule-Oja (UNILAG)':  '#6DB6F2',
    'Falomo (Ikoyi)':      '#2166AC',
    'Ewutuntun (Oshodi)':  '#D85A30',
    'Ajegunle':            '#E07C50',
    'Makoko':              '#C0392B',
    'Obalende':            '#3B6D11',
}

# ── 4. BUILD FIGURE ──────────────────────────────────────────
N      = len(axes_keys)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]

fig = plt.figure(figsize=(14, 9), dpi=600, facecolor='#F7F8FA')
fig.patch.set_facecolor('#F7F8FA')

fig.text(0.5, 0.975,
         'Ward Socio-Environmental Profiles — Radar Analysis',
         ha='center', va='top', fontsize=14, fontweight='bold', color='#1A1A2E')

rows_n, cols_n = 2, 4

for idx, w in enumerate(wards):
    ax    = fig.add_subplot(rows_n, cols_n, idx + 1, projection='polar')
    color = WARD_COLORS.get(w['name'], STATUS_COLOR[w['status']])
    ax.set_facecolor('white')

    norm_vals = [norm(w[k], k) for k in axes_keys]
    values    = norm_vals + [norm_vals[0]]
    raw_vals  = [w[k] for k in axes_keys]

    # Grid rings & spokes
    for r in [0.25, 0.5, 0.75, 1.0]:
        ax.plot(angles, [r] * (N + 1), color='#CCCCDD', linewidth=0.5, zorder=1)
    for a in angles[:-1]:
        ax.plot([a, a], [0, 1], color='#CCCCDD', linewidth=0.5, zorder=1)

    # Data polygon
    ax.fill(angles, values, color=color, alpha=0.22, zorder=2)
    ax.plot(angles, values, color=color, linewidth=1.8, zorder=3)
    ax.scatter(angles[:-1], norm_vals,
               s=22, color=color, zorder=5,
               edgecolors='white', linewidths=0.7)

    # ── Raw value labels at each vertex ──────────────────────
    for i, (angle, nv, rv) in enumerate(zip(angles[:-1], norm_vals, raw_vals)):
        label   = fmt(rv, axes_keys[i])
        r_label = min(nv + 0.16, 1.18)

        x_norm = np.sin(angle)
        if abs(x_norm) < 0.2:
            ha = 'center'
        elif x_norm > 0:
            ha = 'right'
        else:
            ha = 'left'

        ax.text(angle, r_label, label,
                ha=ha, va='center',
                fontsize=5.2, fontweight='bold',
                color=color,
                bbox=dict(boxstyle='round,pad=0.18', facecolor='white',
                          edgecolor=color, linewidth=0.5, alpha=0.88),
                zorder=6)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(axes_labels, fontsize=5.2, color='#333344')
    ax.set_yticks([])
    ax.set_ylim(0, 1.3)
    ax.spines['polar'].set_visible(False)
    ax.set_title(w['name'], pad=14, fontsize=7.5, fontweight='bold', color='#1A1A2E')

# ── 5. LEGEND ────────────────────────────────────────────────
legend_patches = [
    mpatches.Patch(facecolor=STATUS_COLOR['Planned'],   label='Planned'),
    mpatches.Patch(facecolor=STATUS_COLOR['Unplanned'], label='Unplanned'),
    mpatches.Patch(facecolor=STATUS_COLOR['Mixed'],     label='Mixed'),
]
fig.legend(handles=legend_patches, loc='lower center', ncol=3,
           frameon=False, fontsize=8, bbox_to_anchor=(0.5, 0.055),
           title='Planning Status', title_fontsize=8)

# Normalisation note at bottom
fig.text(0.5, 0.022,
         'Axes normalised 0–1 for comparability  ·  MNDWI axis inverted '
         '(higher = drier / more built-up)  ·  Labels show raw values',
         ha='center', va='bottom', fontsize=7.0, color='#666677', style='italic')

plt.subplots_adjust(left=0.04, right=0.96, top=0.91, bottom=0.11,
                    hspace=0.55, wspace=0.45)

# ── 6. SAVE & SHOW ───────────────────────────────────────────
fig.savefig(OUTPUT_PATH, dpi=600, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print(f"Saved to: {OUTPUT_PATH}")

# ── Optional: auto-download in Colab ─────────────────────────
# from google.colab import files
# files.download(OUTPUT_PATH)